In [ ]:
#STEP 1 — Convert agents → tools
tools = [
    web_search_tool,
    sql_query_tool,
    rag_tool
]

In [ ]:
#STEP 2 — Use tool-calling agent
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools)

In [ ]:
#Imports
from typing import TypedDict, List
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sqlalchemy import create_engine, text
from langgraph.graph import StateGraph, END 

In [ ]:
#LLM 
llm = ChatOllama(model="llama3")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
#DB SETUP 
engine = create_engine("sqlite:///example.db")

with engine.begin() as conn:
    conn.execute(text("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER
    )
    """))

    conn.execute(text("DELETE FROM users"))

    conn.execute(text("""
    INSERT INTO users (name, age) VALUES
    ('Ravi', 30),
    ('Anil', 25),
    ('Sita', 28)
    """))

In [ ]:
#RAG SETUP
docs = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation."),
    Document(page_content="Agentic AI refers to systems that can plan and act."),
    Document(page_content="AI trends include automation and autonomous agents.")
]

vectorstore = FAISS.from_documents(docs, embeddings) 


In [ ]:
#SQL TOOL
@tool
def sql_tool(query: str) -> str:
    """Use for database queries about users"""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        return str(result.fetchall())

In [ ]:
#RAG TOOL
@tool
def rag_tool(query: str) -> str:
    """Use for internal knowledge questions"""
    docs = vectorstore.similarity_search(query, k=2)
    return "\n".join([d.page_content for d in docs])

In [ ]:
#WEB TOOL
@tool
def web_tool(query: str) -> str:
    """Use for latest or external information"""
    return llm.invoke(f"Give latest info about {query}").content

In [ ]:
#STATE
class AgentState(TypedDict):
    query: str
    plan: List[str]
    intermediate_results: List[str]
    final_answer: str

In [ ]:
#PLANNING NODE 
def planner(state):
    query = state["query"]

    prompt = f"""
Break the query into steps.

Available tools:
- sql_tool
- rag_tool
- web_tool

Return steps like:
1. tool_name: instruction

Query: {query}
"""

    plan = llm.invoke(prompt).content.split("\n")
    return {"plan": plan, "intermediate_results": []}

In [ ]:
#EXECUTION NODE
def executor(state):
    steps = state["plan"]
    results = []

    for step in steps:
        step = step.lower()

        if "sql_tool" in step:
            res = sql_tool.invoke("SELECT * FROM users")
            results.append(res)

        elif "rag_tool" in step:
            res = rag_tool.invoke(state["query"])
            results.append(res)

        elif "web_tool" in step:
            res = web_tool.invoke(state["query"])
            results.append(res)

    return {"intermediate_results": results}

In [ ]:
#FINAL SYNTHESIS NODE 
def synthesizer(state):
    query = state["query"]
    results = "\n".join(state["intermediate_results"])

    prompt = f"""
Answer the question using results below:

{results}

Question: {query}
"""

    final = llm.invoke(prompt).content
    return {"final_answer": final}

In [ ]:
@LANDGRAPH 

graph = StateGraph(AgentState)

graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("synthesizer", synthesizer)

graph.add_edge("planner", "executor")
graph.add_edge("executor", "synthesizer")
graph.add_edge("synthesizer", END)

graph.set_entry_point("planner")

app = graph.compile()

In [ ]:
#RUN
def run(q):
    result = app.invoke({"query": q})
    print("\nFINAL ANSWER:\n", result["final_answer"])

In [ ]:
#TEST 
run("what is RAG?")
run("show users and explain AI trends")